# 7.0: The Pipeline, Serializable, Reproducible, Not Deployed

**Question this notebook answers:** how is the final model packaged so someone else can
load and reuse it exactly, and what does "not deployed" mean as a deliberate boundary
rather than an unfinished task?

In [1]:
import sys, json
import joblib
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

with open(Path('..') / 'models' / 'model_meta.json') as f:
    meta = json.load(f)

print(json.dumps({k: v for k, v in meta.items() if k not in ('FEATURE_SET', 'X_train_columns', 'hiperparametros')}, indent=2))
print(f"\nFEATURE_SET: {meta['FEATURE_SET_n']} raw feature names -> {len(meta['X_train_columns'])} columns after one-hot encoding")


{
  "model": "XGB_walkforward",
  "FEATURE_SET_n": 79,
  "threshold_operacional": 0.31,
  "data_treino_utc": "2026-07-11T23:55:16.960768+00:00",
  "sklearn_version": "1.9.0",
  "xgboost_version": "3.3.0",
  "python_version": "3.12.10",
  "train_parquet_sha256": "0f7e711edc7c20839c0bd569a0c6f6e3125cadd3dcd1264a57a323b2b89a1191",
  "train_n_rows": 172988
}

FEATURE_SET: 79 raw feature names -> 90 columns after one-hot encoding


## The reproducibility contract

Three conditions, all already enforced in `src/models.py` and `run_all.py`, are what make
the shipped `.joblib` file reproducible from source rather than a one-off artifact:

1. **`random_state=42`** on the estimator itself.
2. **`n_jobs=1`**: this XGBoost build is only bit-for-bit deterministic across separate
   runs at a fixed thread count. Multi-threaded histogram building is not (`notebook 11`
   measured this directly).
3. **Training rows kept in `train.parquet`'s on-disk order**: never re-sorted (e.g., by
   `issue_d`) before fitting. Row order measurably changes XGBoost's histogram splits even
   with an identical seed.

`train_parquet_sha256` above is not decorative. It is the exact file hash the shipped
model was fit on, so a claim of "this reproduces the model" is checkable, not just
asserted.

## Loading the model and scoring new rows

The artifact is a plain `joblib` file, no custom class, no framework lock-in beyond
scikit-learn's estimator interface. Scoring a handful of already-existing test rows shows
the whole usage surface:

In [2]:
from src.data import load_split, FEATURE_SET, CATEGORICAL_COLS
from src.features import build_features, prepare_X

train = load_split('train')
test = load_split('test')
X_train = prepare_X(build_features(train), FEATURE_SET, CATEGORICAL_COLS)
sample = test.sample(5, random_state=0)
X_sample = prepare_X(build_features(sample), FEATURE_SET, CATEGORICAL_COLS).reindex(columns=X_train.columns, fill_value=0)

xgb_final = joblib.load(Path('..') / 'models' / 'xgb_final.joblib')
scored = pd.DataFrame({
    'loan_amnt': sample['loan_amnt'].values,
    'grade': sample['grade'].values,
    'predicted_default_prob': xgb_final.predict_proba(X_sample)[:, 1].round(4),
    'decision_at_0.31': ['reject' if p >= 0.31 else 'approve'
                          for p in xgb_final.predict_proba(X_sample)[:, 1]],
    'actual_outcome': ['Charged Off' if t == 1 else 'Fully Paid' for t in sample['target'].values],
})
scored


,loan_amnt,grade,predicted_default_prob,decision_at_0.31,actual_outcome
0,13925.0,b,0.2401,approve,Fully Paid
1,10000.0,b,0.3001,approve,Fully Paid
2,15000.0,b,0.0464,approve,Fully Paid
3,17000.0,a,0.0111,approve,Fully Paid
4,15000.0,c,0.3140,reject,Charged Off


This is the entire inference surface: `src.features.prepare_X` to encode raw fields
the same way training did, `model.predict_proba`, and a fixed threshold. No feature
computation happens inside the model object. It is entirely reproducible from `src/`
alone (proven end-to-end by `src/verify_pipeline.py`, which retrains from scratch and
checks the test profit reproduces to the cent).

## The single reproduction command

`python run_all.py` runs the essential path (raw CSV to a verified final-model result)
in a fixed order, checking the raw CSV exists before starting and asserting the exact test
profit at the end. Measured on the development machine, CPU only:

| Step | Time |
|---|---|
| Build processed dataset | 75.5s |
| Temporal split | 13.8s |
| Feature engineering | 3.7s |
| Train final model | 55.1s |
| Verify | 39.4s |
| **Total** | **187.6s (~3.1 min)** |

Full details, including what this entry point deliberately does *not* re-run (the
walk-forward tuning and bootstrap notebooks, which take hours), are in `docs/SETUP.md`.

## Not deployed: a scope decision, not a gap

There is no API, no serving container, no monitoring, and no scheduled retraining here.
That is intentional: this project's deliverable is a *validated, reproducible model
artifact and a documented decision process* (`docs/FACTS.md`, `docs/DATA_CARD.md`), not a
production credit system. Shipping an API would invite exactly the misuse
`docs/DATA_CARD.md` warns against: using a model that has only ever seen already-approved
loans (`6.0`) to make real-world approve/reject decisions on general applicants. Packaging
stops at "anyone can load `models/xgb_final.joblib` and get the documented number back",
on purpose.

## Closing the loop

Across these seven notebooks: a population defined by two hard cutoffs (`1.0`), missing
data resolved by mechanism rather than convention (`2.0`), five engineered features that
earned their place and two that didn't (`3.0`), a model chosen by validation across three
time windows instead of one (`4.0`), a calibration check that found recalibration wasn't
free (`5.0`), a profit-first evaluation that surfaces exactly where the model is weaker
than its headline number suggests (`6.0`), and a packaging boundary that stops short of
deployment on purpose (`7.0`, here).

The number that matters most from all of it: **$242.23M in expected profit on the 2015
test set, $9.03M above approving everyone, statistically distinguishable from zero and
from the simplest baseline**, reported alongside every qualification a skeptical reader
would ask for, not instead of them.